# Advanced Prompting Strategies — Hiromi

Comparative evaluation of prompting strategies for hallucination detection on TruthfulQA.

Strategies:
- Zero-shot (baseline)
- Few-shot (baseline)
- Reference-based (baseline)
- Chain-of-Thought (CoT)
- Self-Consistency
- Decomposed Evaluation
- Few-shot optimization (2/4/6/8 examples, category-aware)

In [3]:
import sys
sys.path.insert(0, '..')

In [4]:
from typing import Callable
import os
import time
from pathlib import Path
from dotenv import load_dotenv

import openai
import pandas as pd
from sklearn.metrics import classification_report
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.progress import track

from hiromi.judge.llm import LlmAsAJudge, PromptTemplate
from hiromi.judge.cot import ChainOfThoughtJudge
from hiromi.judge.self_consistency import SelfConsistencyJudge
from hiromi.judge.decomposed import DecomposedJudge
from hiromi.types.judgement import EPrediction

from loguru import logger

load_dotenv()

True

In [5]:
PROMPTS_DIR = Path('../prompts')
DATA_DIR = Path('../data')
YC_MODEL = 'yandexgpt-lite/latest'


def create_client() -> openai.OpenAI:
    return openai.OpenAI(
        api_key=os.getenv('YANDEX_CLOUD_API_KEY'),
        base_url='https://ai.api.cloud.yandex.net/v1',
    )


def create_model(name: str) -> str:
    return f"gpt://{os.getenv('YANDEX_CLOUD_FOLDER')}/{name}"


def parallel_apply(
    df: pd.DataFrame,
    func: Callable[[pd.Series], dict],
    max_workers: int = 5,
) -> pd.DataFrame:
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, row): idx for idx, row in df.iterrows()}
        for future in track(as_completed(futures), total=len(futures)):
            results.append(future.result(timeout=60))
    return pd.DataFrame(results, index=df.index)


def make_report(df: pd.DataFrame, strategy: str = '', latency: float | None = None) -> dict:
    cleaned = df[df['prediction'] != EPrediction.error][['prediction', 'target']]
    report: dict = classification_report(cleaned['target'], cleaned['prediction'], output_dict=True)  # type: ignore
    report['strategy'] = strategy
    report['n_errors'] = int((df['prediction'] == EPrediction.error).sum())
    if latency is not None:
        report['avg_latency_s'] = latency
    return report


def make_pred(sample: pd.Series, judge) -> dict:
    question = sample['question']
    answer = sample['answer']
    t0 = time.perf_counter()
    judgement = judge.predict(question=question, llm_answer=answer)
    elapsed = time.perf_counter() - t0
    pred = sample.to_dict()
    pred.update({
        'prediction': judgement.prediction,
        'full-response': judgement.meta.get('full-model-response', ''),
        'latency_s': elapsed,
    })
    return pred


def make_pred_referenced(sample: pd.Series, judge: LlmAsAJudge) -> dict:
    question = sample['question']
    answer = sample['answer']
    correct_examples = sample['correct_examples']
    incorrect_examples = sample['incorrect_examples']
    t0 = time.perf_counter()
    judgement = judge.predict(
        question=question, llm_answer=answer,
        correct_examples=correct_examples, incorrect_examples=incorrect_examples,
    )
    elapsed = time.perf_counter() - t0
    pred = sample.to_dict()
    pred.update({
        'prediction': judgement.prediction,
        'full-response': judgement.meta.get('full-model-response', ''),
        'latency_s': elapsed,
    })
    return pred

## Data Preparation

In [6]:
data = pd.read_csv(DATA_DIR / 'TruthfulQA.csv')


def process_row(row: pd.Series) -> pd.DataFrame:
    rows = [
        {'question': row['Question'], 'answer': row['Best Answer'], 'target': EPrediction.correct, 'is_best': True, 'type': row['Type'], 'category': row['Category']},
        {'question': row['Question'], 'answer': row['Best Incorrect Answer'], 'target': EPrediction.incorrect, 'is_best': True, 'type': row['Type'], 'category': row['Category']},
    ]
    for ans in row['Correct Answers'].split(';'):
        rows.append({'question': row['Question'], 'answer': ans.strip(), 'target': EPrediction.correct, 'is_best': False, 'type': row['Type'], 'category': row['Category']})
    for ans in row['Incorrect Answers'].split(';'):
        rows.append({'question': row['Question'], 'answer': ans.strip(), 'target': EPrediction.incorrect, 'is_best': False, 'type': row['Type'], 'category': row['Category']})
    return pd.DataFrame(rows)


def process_row_referenced(row: pd.Series) -> pd.DataFrame:
    correct_list = ([row['Best Answer']] if pd.notna(row['Best Answer']) else []) + \
                   [a.strip() for a in row['Correct Answers'].split(';') if a.strip()]
    incorrect_list = ([row['Best Incorrect Answer']] if pd.notna(row['Best Incorrect Answer']) else []) + \
                     [a.strip() for a in row['Incorrect Answers'].split(';') if a.strip()]
    rows = []
    for ans in correct_list:
        rows.append({
            'question': row['Question'], 'answer': ans, 'target': EPrediction.correct,
            'correct_examples': '; '.join(a for a in correct_list if a != ans),
            'incorrect_examples': '; '.join(incorrect_list),
            'is_best': ans == row.get('Best Answer'), 'type': row['Type'], 'category': row['Category'],
        })
    for ans in incorrect_list:
        rows.append({
            'question': row['Question'], 'answer': ans, 'target': EPrediction.incorrect,
            'correct_examples': '; '.join(correct_list),
            'incorrect_examples': '; '.join(a for a in incorrect_list if a != ans),
            'is_best': ans == row.get('Best Incorrect Answer'), 'type': row['Type'], 'category': row['Category'],
        })
    return pd.DataFrame(rows)


dataset = pd.concat(data.apply(process_row, axis=1).tolist(), ignore_index=True)  # type: ignore
dataset_referenced = pd.concat(data.apply(process_row_referenced, axis=1).tolist(), ignore_index=True)  # type: ignore
print(f'dataset: {len(dataset)} rows | dataset_referenced: {len(dataset_referenced)} rows')

dataset: 7632 rows | dataset_referenced: 7608 rows


## Chain-of-Thought (CoT)

In [ ]:
cot_judge = ChainOfThoughtJudge(
    prompt=PromptTemplate.from_file(PROMPTS_DIR / 'cot.txt'),
    client=create_client(),
    model=create_model(YC_MODEL),
)

t_start = time.perf_counter()
cot_predict = parallel_apply(dataset, lambda x: make_pred(x, cot_judge))
cot_elapsed = time.perf_counter() - t_start

cot_predict.to_csv(DATA_DIR / 'cot_predict.csv', index=False)
cot_report = make_report(cot_predict, strategy='cot', latency=cot_elapsed / len(cot_predict))
print(f"CoT accuracy: {cot_report['accuracy']:.4f}  F1: {cot_report['weighted avg']['f1-score']:.4f}")

Output()

## Self-Consistency

In [ ]:
# Self-consistency is 5× more expensive — use CoT as the base judge
base_judge = ChainOfThoughtJudge(
    prompt=PromptTemplate.from_file(PROMPTS_DIR / 'cot.txt'),
    client=create_client(),
    model=create_model(YC_MODEL),
)
sc_judge = SelfConsistencyJudge(base_judge=base_judge, n_samples=5, temperature=0.5)

t_start = time.perf_counter()
sc_predict = parallel_apply(dataset, lambda x: make_pred(x, sc_judge), max_workers=3)
sc_elapsed = time.perf_counter() - t_start

sc_predict.to_csv(DATA_DIR / 'self_consistency_predict.csv', index=False)
sc_report = make_report(sc_predict, strategy='self_consistency', latency=sc_elapsed / len(sc_predict))
print(f"Self-Consistency accuracy: {sc_report['accuracy']:.4f}  F1: {sc_report['weighted avg']['f1-score']:.4f}")

## Decomposed Evaluation

In [ ]:
decomposed_judge = DecomposedJudge(
    client=create_client(),
    model=create_model(YC_MODEL),
    factual_prompt=PromptTemplate.from_file(PROMPTS_DIR / 'decomposed_factual.txt'),
    verification_prompt=PromptTemplate.from_file(PROMPTS_DIR / 'decomposed_verification.txt'),
    fabrication_prompt=PromptTemplate.from_file(PROMPTS_DIR / 'decomposed_fabrication.txt'),
)

t_start = time.perf_counter()
decomposed_predict = parallel_apply(dataset, lambda x: make_pred(x, decomposed_judge), max_workers=3)
decomposed_elapsed = time.perf_counter() - t_start

decomposed_predict.to_csv(DATA_DIR / 'decomposed_predict.csv', index=False)
decomposed_report = make_report(decomposed_predict, strategy='decomposed', latency=decomposed_elapsed / len(decomposed_predict))
print(f"Decomposed accuracy: {decomposed_report['accuracy']:.4f}  F1: {decomposed_report['weighted avg']['f1-score']:.4f}")

## Few-Shot Optimization

In [ ]:
# All 8 generic examples (subset used based on n_examples parameter)
ALL_EXAMPLES = [
    ('What is the capital of France?', 'Paris is the capital of France.', 'correct'),
    ('Who wrote the novel "Pride and Prejudice"?', 'The novel was written by Charles Dickens.', 'hallucination'),
    ('What is the chemical symbol for gold?', 'The chemical symbol for gold is Au, from the Latin "aurum".', 'correct'),
    ('When did World War II end?', 'World War II ended in 1945, specifically on September 2nd when Japan surrendered.', 'correct'),
    ('How many planets are in our solar system?', 'There are nine planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune, and Pluto.', 'hallucination'),
    ('What is the largest ocean on Earth?', 'The largest ocean is the Atlantic Ocean.', 'hallucination'),
    ('What is the speed of light in a vacuum?', 'The speed of light is approximately 300,000 kilometres per second.', 'correct'),
    ('Who painted the Mona Lisa?', 'The Mona Lisa was painted by Michelangelo.', 'hallucination'),
]


def build_few_shot_prompt(n_examples: int, examples: list | None = None) -> str:
    if examples is None:
        examples = ALL_EXAMPLES[:n_examples]
    header = (
        'You are an expert evaluator of factual consistency. Determine whether the answer is '
        'factually correct or a hallucination.\n\n'
        'Instructions:\n'
        '- Output exactly: correct\n'
        '- Or output exactly: hallucination\n'
        '- Do not include any additional text.\n\n'
        '--- Examples ---\n'
    )
    body = ''
    for i, (q, a, label) in enumerate(examples, 1):
        body += f'Example {i}:\nQuestion: {q}\nAnswer: {a}\nOutput: {label}\n\n'
    footer = '--- Now evaluate ---\n\nQuestion:\n{question}\n\nAnswer to evaluate:\n{llm_answer}\n'
    return header + body + footer


fs_reports = {}
for n in [2, 4, 6, 8]:
    judge = LlmAsAJudge(
        prompt=build_few_shot_prompt(n),
        client=create_client(),
        model=create_model(YC_MODEL),
    )
    t0 = time.perf_counter()
    preds = parallel_apply(dataset, lambda x: make_pred(x, judge))
    elapsed = time.perf_counter() - t0
    preds.to_csv(DATA_DIR / f'few_shot_{n}_predict.csv', index=False)
    fs_reports[f'few_shot_{n}'] = make_report(preds, strategy=f'few_shot_{n}', latency=elapsed / len(preds))
    print(f"Few-shot ({n} examples) accuracy: {fs_reports[f'few_shot_{n}']['accuracy']:.4f}  F1: {fs_reports[f'few_shot_{n}']['weighted avg']['f1-score']:.4f}")

### Category-Aware Few-Shot

In [ ]:
# Build category-specific example pools from the dataset itself
# Use only 'Best Answer' (correct) and 'Best Incorrect Answer' (hallucination) as examples

def build_category_examples(data: pd.DataFrame, n_per_label: int = 3) -> dict[str, list]:
    """Build per-category example pools from the raw TruthfulQA data."""
    category_examples: dict[str, list] = {}
    for _, row in data.iterrows():
        cat = row['Category']
        if cat not in category_examples:
            category_examples[cat] = []
        category_examples[cat].append((
            row['Question'], row['Best Answer'], 'correct'
        ))
        category_examples[cat].append((
            row['Question'], row['Best Incorrect Answer'], 'hallucination'
        ))
    return category_examples


category_examples = build_category_examples(data)


def make_pred_category_aware(sample: pd.Series, n_examples: int = 6) -> dict:
    cat = sample['category']
    pool = category_examples.get(cat, [])
    # Exclude examples where the question matches the current sample (no data leakage)
    pool = [(q, a, l) for q, a, l in pool if q != sample['question']]
    # Balance: take n_examples/2 correct + n_examples/2 hallucination
    correct_ex = [(q, a, l) for q, a, l in pool if l == 'correct'][:n_examples // 2]
    incorrect_ex = [(q, a, l) for q, a, l in pool if l == 'hallucination'][:n_examples // 2]
    examples = correct_ex + incorrect_ex
    # Fall back to generic examples if category pool is too small
    if len(examples) < n_examples:
        examples = ALL_EXAMPLES[:n_examples]

    prompt_text = build_few_shot_prompt(n_examples, examples)
    judge = LlmAsAJudge(
        prompt=prompt_text,
        client=create_client(),
        model=create_model(YC_MODEL),
    )
    return make_pred(sample, judge)


t0 = time.perf_counter()
category_aware_predict = parallel_apply(dataset, make_pred_category_aware, max_workers=5)
cat_elapsed = time.perf_counter() - t0

category_aware_predict.to_csv(DATA_DIR / 'category_aware_few_shot_predict.csv', index=False)
cat_report = make_report(category_aware_predict, strategy='category_aware_few_shot', latency=cat_elapsed / len(category_aware_predict))
print(f"Category-aware few-shot accuracy: {cat_report['accuracy']:.4f}  F1: {cat_report['weighted avg']['f1-score']:.4f}")

## Comparative Analysis

In [ ]:
# Load pre-computed baseline results
zero_shot_predict = pd.read_csv(DATA_DIR / 'zero_shot_predict.csv')
few_shot_predict = pd.read_csv(DATA_DIR / 'few_shot_predict.csv')
referenced_predict = pd.read_csv(DATA_DIR / 'referenced_llm_predict.csv')

# Convert prediction columns back to EPrediction (stored as int in CSV)
for df in [zero_shot_predict, few_shot_predict, referenced_predict,
           cot_predict, sc_predict, decomposed_predict, category_aware_predict]:
    df['prediction'] = df['prediction'].map(lambda x: EPrediction(int(x)) if pd.notna(x) else EPrediction.error)

baseline_zero = make_report(zero_shot_predict, strategy='zero_shot')
baseline_few = make_report(few_shot_predict, strategy='few_shot_6')
baseline_ref = make_report(referenced_predict, strategy='reference_based')

In [ ]:
all_reports = [
    baseline_zero,
    baseline_few,
    baseline_ref,
    cot_report,
    sc_report,
    decomposed_report,
    cat_report,
    *fs_reports.values(),
]

summary_rows = []
for r in all_reports:
    summary_rows.append({
        'strategy': r.get('strategy', ''),
        'accuracy': round(r.get('accuracy', float('nan')), 4),
        'precision': round(r.get('weighted avg', {}).get('precision', float('nan')), 4),
        'recall': round(r.get('weighted avg', {}).get('recall', float('nan')), 4),
        'f1': round(r.get('weighted avg', {}).get('f1-score', float('nan')), 4),
        'n_errors': r.get('n_errors', '?'),
        'avg_latency_s': round(r.get('avg_latency_s', float('nan')), 3),
    })

summary = pd.DataFrame(summary_rows).sort_values('accuracy', ascending=False)
summary

In [ ]:
# Per-category accuracy breakdown
def category_accuracy(df: pd.DataFrame, strategy: str) -> pd.DataFrame:
    cleaned = df[df['prediction'] != EPrediction.error].copy()
    cleaned['correct'] = cleaned['prediction'] == cleaned['target']
    acc = cleaned.groupby('category')['correct'].mean().rename(strategy)
    return acc


cat_breakdown = pd.DataFrame({
    'zero_shot': category_accuracy(zero_shot_predict, 'zero_shot'),
    'few_shot_6': category_accuracy(few_shot_predict, 'few_shot_6'),
    'reference_based': category_accuracy(referenced_predict, 'reference_based'),
    'cot': category_accuracy(cot_predict, 'cot'),
    'self_consistency': category_accuracy(sc_predict, 'self_consistency'),
    'decomposed': category_accuracy(decomposed_predict, 'decomposed'),
    'category_aware': category_accuracy(category_aware_predict, 'category_aware'),
})

# Best strategy per category
cat_breakdown['best_strategy'] = cat_breakdown.idxmax(axis=1)
cat_breakdown = cat_breakdown.sort_values('reference_based', ascending=False)
cat_breakdown

In [ ]:
summary.to_csv(DATA_DIR / 'strategy_comparison.csv', index=False)
cat_breakdown.to_csv(DATA_DIR / 'category_breakdown.csv')
print('Results saved to data/strategy_comparison.csv and data/category_breakdown.csv')